[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [3]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [4]:
import torch
import math

In [15]:
# ✏️ YOUR IMPLEMENTATION HERE

def rotate_pair(x, y, theta):
    x, y = x * torch.cos(theta) - y * torch.sin(theta), x * torch.sin(theta) + y * torch.cos(theta)
    return x,y

def split_into_pairs(x):
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return x1, x2

def create_theta(theta, max_pos):
    pos_theta = torch.stack([theta * i for i in range(max_pos)], dim=0)
    return pos_theta

def apply_rope(q, k):
    B, S, D = k.shape
    theta = torch.tensor([10000**(-2.0*i / D) for i in range(D//2)])
    pos_theta = create_theta(theta, S)
    k1, k2  = split_into_pairs(k)
    q1, q2  = split_into_pairs(q)
    rk1, rk2 = rotate_pair(k1, k2, theta)
    rq1, rq2 = rotate_pair(q1, q2, theta)
    rq = torch.cat([rq1, rq2], dim=-1)
    rk = torch.cat([rk1, rk2], dim=-1)
    return rq, rk

    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    pass

In [16]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [17]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (1.1ms)
  ✅ [2/4] Preserves norm (5.4ms)
  ✅ [3/4] Relative position property (7.3ms)
  ✅ [4/4] Gradient flow (15.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (29.5ms total)
  Progress saved. Run status() to see your dashboard.



In [26]:
# ✏️ YOUR IMPLEMENTATION HERE

def rotate_pair(x, y, theta):
    cosTheta = torch.cos(theta)
    sinTheta = torch.sin(theta)
    x1  = torch.einsum('bsd,sd->bsd', x, cosTheta)
    x2  = torch.einsum('bsd,sd->bsd', x, sinTheta)
    y1  = torch.einsum('bsd,sd->bsd', y, cosTheta)
    y2  = torch.einsum('bsd,sd->bsd', y, sinTheta)
    x, y = x1 - y2, x2 + y1
    return x,y

def split_into_pairs(x):
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return x1, x2

def create_theta(theta, max_pos):
    pos_theta = torch.stack([theta * i for i in range(max_pos)], dim=0)
    return pos_theta

def apply_rope(q, k):
    B, S, D = k.shape
    B1, S1, D1 = q.shape
    theta = torch.tensor([10000**(-2.0*i / D) for i in range(D//2)])
    theta1 = torch.tensor([10000**(-2.0*i / D1) for i in range(D1//2)])
    pos_theta = create_theta(theta, S)
    pos_theta1 = create_theta(theta1, S1)
    k1, k2  = split_into_pairs(k)
    q1, q2  = split_into_pairs(q)
    rk1, rk2 = rotate_pair(k1, k2, pos_theta)
    rq1, rq2 = rotate_pair(q1, q2, pos_theta1)
    rq = torch.cat([rq1, rq2], dim=-1)
    rk = torch.cat([rk1, rk2], dim=-1)
    return rq, rk

    # 1. Compute position angles
    # 2. Split into even/odd pairs
    # 3. Apply rotation
    pass

In [27]:
# same content at every position; scores must depend only on (m - n)
qc = q[:, :1].expand(-1, 8, -1).contiguous()
kc = k[:, :1].expand(-1, 8, -1).contiguous()
qr, kr = apply_rope(qc, kc)
s = torch.einsum('bsd,btd->bst', qr, kr)[0]
print('Relative only:', all(abs(s[i,j] - s[i+1,j+1]) < 1e-4 for i in range(7) for j in range(7)))
print('Positions distinguishable:', not torch.allclose(qr[0,0], qr[0,3], atol=1e-6))

Relative only: True
Positions distinguishable: True


In [28]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (1.7ms)
  ✅ [2/4] Preserves norm (4.0ms)
  ✅ [3/4] Relative position property (2.6ms)
  ✅ [4/4] Gradient flow (1.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (9.4ms total)
  Progress saved. Run status() to see your dashboard.



In [38]:
def rotate_pair(x, y, theta):
    cosTheta = torch.cos(theta)
    sinTheta = torch.sin(theta)
    x1  = torch.einsum('bsd,sd->bsd', x, cosTheta)
    x2  = torch.einsum('bsd,sd->bsd', x, sinTheta)
    y1  = torch.einsum('bsd,sd->bsd', y, cosTheta)
    y2  = torch.einsum('bsd,sd->bsd', y, sinTheta)
    x, y = x1 - y2, x2 + y1
    return x,y

def split_into_pairs(x):
    x1, x2 = torch.chunk(x, 2, dim=-1)
    return x1, x2

def apply_rope(q, k, q_offset=0):
    B, S, D = k.shape
    inv_freq = 10000 ** (-torch.arange(0, D, 2) / 2)

    pos_k = torch.arange(S)
    pos_q = torch.arange(q.shape[1]) + q_offset #(if any)

    theta_k = torch.outer(pos_k, inv_freq)
    theta_q = torch.outer(pos_q, inv_freq)

    q1, q2 = split_into_pairs(q)
    k1, k2 = split_into_pairs(k)

    rq = torch.cat(rotate_pair(q1, q2, theta_q), dim=-1)
    rk = torch.cat(rotate_pair(k1, k2, theta_k), dim=-1)
    return rq, rk

In [39]:
# same content at every position; scores must depend only on (m - n)
qc = q[:, :1].expand(-1, 8, -1).contiguous()
kc = k[:, :1].expand(-1, 8, -1).contiguous()
qr, kr = apply_rope(qc, kc)
s = torch.einsum('bsd,btd->bst', qr, kr)[0]
print('Relative only:', all(abs(s[i,j] - s[i+1,j+1]) < 1e-4 for i in range(7) for j in range(7)))
print('Positions distinguishable:', not torch.allclose(qr[0,0], qr[0,3], atol=1e-6))

Relative only: True
Positions distinguishable: True


In [40]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (1.7ms)
  ✅ [2/4] Preserves norm (4.5ms)
  ✅ [3/4] Relative position property (4.3ms)
  ✅ [4/4] Gradient flow (1.7ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (12.3ms total)
  Progress saved. Run status() to see your dashboard.

